# 01 · EDA y calidad de datos (tarea A2)

**Fase CRISP-DM:** comprensión de los datos. Dataset Rossmann Store Sales (Kaggle), 2013-01-01 a
2015-07-31. Debajo de cada gráfico hay una conclusión escrita.

Regla: `Customers` se mira solo de forma descriptiva; **nunca** es variable exógena (fuga de datos).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import DATA_PROCESSED, LEVELS_TYPES
from src.data.load import load_raw

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:,.2f}".format)

PROCESSED = DATA_PROCESSED / "rossmann.parquet"
if not PROCESSED.exists():
    from src.data.make_dataset import main as make_dataset
    make_dataset()

train_raw, store_raw = load_raw()
df = pd.read_parquet(PROCESSED)
obs = df[df["imputed"] == 0]          # solo registros originales
opened = obs[obs["Open"] == 1]
print(train_raw.shape, store_raw.shape, df.shape)

## 1. Revisión inicial

In [ ]:
display(train_raw.dtypes.to_frame("dtype").T)
display(store_raw.dtypes.to_frame("dtype").T)
print("Rango de fechas:", train_raw["Date"].min().date(), "→", train_raw["Date"].max().date())
print("Tiendas:", train_raw["Store"].nunique(), "| días:", train_raw["Date"].nunique())
print("Valores de StateHoliday en el CSV:", train_raw["StateHoliday"].unique())
print("Nulos en store.csv:")
display(store_raw.isna().sum().to_frame("nulos").T)

In [ ]:
stores_per_day = train_raw.groupby("Date")["Store"].nunique()
ax = stores_per_day.plot(figsize=(11, 3), title="Tiendas con registro por día")
ax.set_ylabel("tiendas"); plt.show()
gap = stores_per_day[stores_per_day < stores_per_day.max()]
print("Días con tiendas faltantes:", len(gap), "|", gap.index.min().date(), "→", gap.index.max().date())
print("Tiendas afectadas:", stores_per_day.max() - gap.min())

> **Conclusión:** _completar después de ejecutar._

## 2. Resumen estadístico de Sales (días abiertos)

In [ ]:
for col in ["StoreType", "Assortment", "DayOfWeek", "Promo", "StateHoliday"]:
    print(f"— por {col}")
    display(opened.groupby(col)["Sales"].describe()[["count", "mean", "50%", "std", "min", "max"]])

> **Conclusión:** _completar después de ejecutar._

## 3. Outliers

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.boxplot(data=opened, x="StoreType", y="Sales", order=list("abcd"), ax=ax, fliersize=1)
ax.set_title("Ventas diarias por tienda y tipo (días abiertos)"); plt.show()

total = df.groupby("Date")["Sales"].sum()
print("Top 10 días más altos (total):"); display(total.nlargest(10).to_frame("Sales"))
print("Top 10 días más bajos con tiendas abiertas:")
open_days = df.groupby("Date")["Open"].mean() > 0.5
display(total[open_days].nsmallest(10).to_frame("Sales"))

> **Conclusión:** _completar después de ejecutar._

## 4. Visualización

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.5))
total.plot(ax=ax, lw=0.6, label="diaria")
total.rolling(28).mean().plot(ax=ax, lw=2, label="media móvil 28 días")
ax.set_title("Ventas totales diarias"); ax.legend(); plt.show()

> **Conclusión:** _completar después de ejecutar._

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
sns.histplot(opened["Sales"], bins=80, ax=axes[0]); axes[0].set_title("Sales")
sns.histplot(np.log1p(opened["Sales"]), bins=80, ax=axes[1]); axes[1].set_title("log1p(Sales)")
plt.show()

> **Conclusión:** _completar después de ejecutar._

In [ ]:
dias = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
heat = (opened.assign(dow=opened["Date"].dt.dayofweek, mes=opened["Date"].dt.month)
        .pivot_table(index="dow", columns="mes", values="Sales", aggfunc="mean"))
heat.index = [dias[i] for i in heat.index]
plt.figure(figsize=(11, 3.8))
sns.heatmap(heat, cmap="Blues", fmt=",.0f", annot=True, annot_kws={"size": 7})
plt.title("Venta promedio por tienda: día de la semana × mes"); plt.show()

> **Conclusión:** _completar después de ejecutar._

In [ ]:
promo = opened.groupby(["StoreType", "Promo"])["Sales"].mean().unstack()
promo.columns = ["Sin promo", "Con promo"]
promo["uplift_%"] = (promo["Con promo"] / promo["Sin promo"] - 1) * 100
display(promo)
promo[["Sin promo", "Con promo"]].plot.bar(figsize=(7, 3.5), rot=0,
                                            title="Venta media con y sin promo por StoreType")
plt.show()

> **Conclusión:** _completar después de ejecutar._

In [ ]:
sample = opened.sample(min(20_000, len(opened)), random_state=42)
sns.scatterplot(data=sample, x="Customers", y="Sales", hue="StoreType", s=5, alpha=0.4)
plt.title("Sales vs Customers (solo descriptivo: Customers NO es exógena)"); plt.show()
print("Correlación:", round(opened[["Sales", "Customers"]].corr().iat[0, 1], 3))

> **Conclusión:** _completar después de ejecutar._

## 5. Anomalías y efecto del relleno del hueco (D-07, D-10)

In [ ]:
print("Días abiertos con ventas 0:", int(obs["anomaly_open_zero"].sum()),
      f"({obs['anomaly_open_zero'].mean() * 100:.3f} % de las filas)")
print("Filas imputadas:", int(df["imputed"].sum()), "| tiendas:", df.loc[df.imputed == 1, "Store"].nunique())
fig, ax = plt.subplots(figsize=(12, 3.5))
obs.groupby("Date")["Sales"].sum().rolling(7).mean().plot(ax=ax, label="sin relleno")
total.rolling(7).mean().plot(ax=ax, label="con relleno (D-10)")
ax.set_title("Total semanal móvil: efecto del relleno del hueco de 2014"); ax.legend(); plt.show()

> **Conclusión:** _completar después de ejecutar._

## 6. Checklist de calidad ISO/IEC 25012

In [ ]:
iso = pd.DataFrame([
    ["Exactitud", "Días abiertos con Sales = 0", int(obs["anomaly_open_zero"].sum())],
    ["Exactitud", "Ventas negativas", int((obs["Sales"] < 0).sum())],
    ["Completitud", "Pares (tienda, fecha) faltantes", int(df["imputed"].sum())],
    ["Completitud", "Nulos en CompetitionDistance", int(store_raw["CompetitionDistance"].isna().sum())],
    ["Completitud", "Nulos en CompetitionOpenSince*", int(store_raw["CompetitionOpenSinceYear"].isna().sum())],
    ["Consistencia", "Tipos distintos en StateHoliday (crudo)", int(train_raw["StateHoliday"].map(type).nunique())],
    ["Consistencia", "Cerrado con ventas > 0", int(((obs["Open"] == 0) & (obs["Sales"] > 0)).sum())],
    ["Consistencia", "Duplicados (Store, Date)", int(train_raw.duplicated(["Store", "Date"]).sum())],
    ["Credibilidad", "Días con venta total > p99.9", int((total > total.quantile(0.999)).sum())],
    ["Actualidad", "Última fecha", str(train_raw["Date"].max().date())],
], columns=["Dimensión", "Chequeo", "Resultado"])
iso

> **Conclusión:** _completar después de ejecutar._

## 7. Selección de las 5 tiendas (`SELECTED_STORES`)

Criterio: una por StoreType con historia completa (sin imputación ni días abiertos con venta 0) y venta cercana a la mediana de su tipo, más una con alta respuesta a la promo.

In [ ]:
complete = obs.groupby("Store").agg(n=("Date", "nunique"), anom=("anomaly_open_zero", "sum"))
complete = complete[(complete["n"] == obs["Date"].nunique()) & (complete["anom"] == 0)].index

wk = opened[opened["Date"].dt.dayofweek < 5]
uplift = (wk.groupby(["Store", "Promo"])["Sales"].mean().unstack()
            .pipe(lambda t: t[1] / t[0] - 1).rename("uplift"))
info = (opened.groupby("Store").agg(sales=("Sales", "mean"), StoreType=("StoreType", "first"))
          .join(uplift).loc[lambda t: t.index.isin(complete)])

chosen = []
for t in "abcd":
    g = info[info["StoreType"] == t]
    if g.empty:
        print(f"Tipo {t}: ninguna tienda con historia completa")
        continue
    chosen.append(int((g["sales"] - g["sales"].median()).abs().idxmin()))
rest = info.drop(index=chosen)["uplift"].dropna()
if len(rest):
    chosen.append(int(rest.idxmax()))  # la de mayor respuesta a la promo
display(info.loc[chosen])
print("Propuesta SELECTED_STORES =", chosen)

> **Conclusión:** _completar_. Proponer el cambio en `src/config.py` (archivo compartido) y regenerar `history.parquet` con `python -m src.features.exog`.

## 8. Validación de D-01 (domingos y `open_share`)

In [ ]:
sun = df[df["Date"].dt.dayofweek == 6]
print(f"Tiendas abiertas en domingo: {sun['Open'].mean() * 100:.2f} % de tienda-días")
print("Por tipo:"); display(sun.groupby("StoreType")["Open"].mean().to_frame("open_share domingo"))
share = df.groupby(df["Date"].dt.dayofweek)["Sales"].sum() / df["Sales"].sum() * 100
display(share.rename(index=dict(enumerate(dias))).to_frame("% de la venta"))

> **Conclusión D-01:** _completar_. Si casi ningún tipo abre en domingo, la serie diaria con
> domingos ≈ 0 se modela bien con s = 7 + `open_share` (conocida de antemano). Si se cambia D-01,
> aplicar el protocolo 6.2 porque afecta al contrato.